<a href="https://colab.research.google.com/github/Towa-1103/Experiment/blob/main/res01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. 必要なライブラリのみをインストール（torchは除外）
!pip install -q -U transformers accelerate sentence-transformers


# 2. LLM（Qwen-2.5-3B-Instruct）の読み込み
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-3B-Instruct"
print(f"[{model_id}] の読み込みを開始します...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,  # メモリ節約（最新仕様に対応）
    device_map="auto",  # GPUへ自動配置
)

# 正常に読み込めたかの確認
print("\n✅ セットアップ完了")
print(f"・使用デバイス: {model.device}")
if torch.cuda.is_available():
  print(f"・GPU名: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 59.1 MB/s eta 0:00:00
[Qwen/Qwen2.5-3B-Instruct] の読み込みを開始します...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ セットアップ完了
・使用デバイス: cuda:0
・GPU名: Tesla T4


In [2]:
import os

# 必ず /content を基準にする
%cd /content

REPO_URL = "https://github.com//Experiment.git"
TARGET_DIR = "/content/Experiment"

if not os.path.exists(TARGET_DIR):
  !git clone {REPO_URL}
  %cd {TARGET_DIR}
else:
  %cd {TARGET_DIR}
  !git pull

print("\n 現在地:", os.getcwd())
print("\n リポジトリの同期が完了しました。")

/content
/content/Experiment
Already up to date.

 現在地: /content/Experiment

 リポジトリの同期が完了しました。


In [13]:
# 1. VS Codeでプッシュした最新コードを取得・再読み込み
!git pull
import importlib
import dialogue_processor

importlib.reload(dialogue_processor)
from dialogue_processor import extract_memories_from_log
from memory_manager import MemoryManager

# 2. ログファイル名を合わせる
LOG_FILE = "Friend_A_2025.txt"
MEMORY_FILE = "memories.json"

manager = MemoryManager(MEMORY_FILE)
print(f"処理前の総記憶数: {len(manager.get_all())} 件")

with open(LOG_FILE, "r", encoding="utf-8") as f:
  raw_log = f.read()

# 3. 実行
print(f"\n'{LOG_FILE}' から記憶を抽出中...")
extracted = extract_memories_from_log(
    raw_log, tokenizer, model, min_importance=3
)
print(f"抽出された候補: {len(extracted)} 件")

# 4. 保存
added_count = manager.add_memories(extracted)
print(f"新規追記: {added_count} 件")
print(f"処理後の総記憶数: {len(manager.get_all())} 件\n")

# 5. 中身の確認
!cat {MEMORY_FILE}

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.18 KiB | 402.00 KiB/s, done.
From https://github.com/Towa-1103/Experiment
   85906be..25ab73b  main       -> origin/main
Updating 85906be..25ab73b
Fast-forward
 dialogue_processor.py | 54 ++++++++++++++++++++++++++++++++-------------------
 1 file changed, 34 insertions(+), 20 deletions(-)
処理前の総記憶数: 0 件

'Friend_A_2025.txt' から記憶を抽出中...
抽出された候補: 12 件
新規追記: 1 件
処理後の総記憶数: 1 件

[
  {
    "timestamp": "2025-11-25 15:39:00",
    "sender_id": "friend",
    "text": "友人A 12/3に顔合わせやるらしい",
    "past_reply": "",
    "importance": 8,
    "id": 1,
    "created_at": "2026-09-25 10:31:36"
  }
]

In [12]:
import torch

# 1. テキストファイルの中身確認
with open("Friend_A_2025.txt", "r", encoding="utf-8") as f:
  raw_log = f.read()

print("=== [Friend_A_2025.txt の中身（先頭200文字）] ===")
print(raw_log[:200])
print("==========================================\n")

# 2. LLMの生の出力確認
import dialogue_processor
from dialogue_processor import EXTRACT_ALL_PROMPT

prompt = EXTRACT_ALL_PROMPT.format(dialogue=raw_log.strip())
messages = [
    {
        "role": "system",
        "content": (
            "あなたは対話ログから有益な記憶を網羅的に抽出し、正確なJSON配列のみを出力するデータ処理エンジンです。"
        ),
    },
    {"role": "user", "content": prompt},
]
text_input = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(text_input, return_tensors="pt").to(model.device)

with torch.no_grad():
  outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False)

raw_response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
)

print("=== [LLMの生の返答] ===")
print(raw_response)
print("=======================")

=== [Friend_A_2025.txt の中身（先頭200文字）] ===
2025.11.25 火曜日
15:39 友人A 12/3に顔合わせやるらしい
15:40 友人A 全員やります
17:20 自分 やったりましょう
17:20 自分 何すんだろ
18:10 友人A 居酒屋行く流れになったら帰宅
18:11 友人A 酔った知り合いAに逆に殺られる
18:11 友人A 普通に自己紹介とかするんかな
20:13 自分 なんだろ
20:14 自分 福永そういうタイプだっ

=== [LLMの生の返答] ===
[
  {
    "timestamp": "2025-11-25 15:39:00",
    "sender_id": "friend",
    "text": "友人A 12/3に顔合わせやるらしい",
    "past_reply": "",
    "importance": 7
  },
  {
    "timestamp": "2025-11-25 15:40:00",
    "sender_id": "friend",
    "text": "友人A 全員やります",
    "past_reply": "",
    "importance": 7
  },
  {
    "timestamp": "2025-11-25 17:20:00",
    "sender_id": "self",
    "text": "やったりしましょう",
    "past_reply": "何すんだろ",
    "importance": 8
  },
  {
    "timestamp": "2025-11-25 18:10:00",
    "sender_id": "friend",
    "text": "居酒屋行く流れになったら帰宅",
    "past_reply": "",
    "importance": 7
  },
  {
    "timestamp": "2025-11-25 18:11:00",
    "sender_id": "friend",
    "text": "酔った知り合いAに逆に殺られる",
    "past_reply": "",
    "importance": 8
  